# Preprocess the Alligned Dataset

In [21]:
import pandas as pd

data = pd.read_parquet("../src/processed_data/notebook_data/alligned_data.parquet")
data.head()

,region,time_ref,time,lt,ws10m_mean,ws10m_std,rh2m_mean,rh2m_std,t2m_mean,t2m_std,...,mslp_std,air_temperature_2m,air_pressure_at_sea_level,relative_humidity_2m,wind_speed_10m,capacity_total,precipitation_amount,wd10m_mean,wind_direction_10m,power_MW
0,NO1,2020-02-15 12:00:00,2020-02-15 12:00:00,0,2.885023,0.564210,0.849861,0.049327,275.620253,0.453982,...,39.606556,275.814952,100545.634126,0.861225,2.652674,405.7,0.001116,218.406386,214.646028,72.052233
1,NO1,2020-02-15 12:00:00,2020-02-15 13:00:00,1,3.233010,0.406024,0.838769,0.050891,275.693526,0.425750,...,28.523728,275.814952,100545.634126,0.861225,2.652674,405.7,0.001116,217.522286,214.646028,44.666173
2,NO1,2020-02-15 12:00:00,2020-02-15 14:00:00,2,3.101949,0.434790,0.862924,0.055345,275.377810,0.414945,...,43.219588,275.814952,100545.634126,0.861225,2.652674,405.7,0.001116,204.505754,214.646028,65.059451
3,NO1,2020-02-15 12:00:00,2020-02-15 15:00:00,3,2.913593,0.367008,0.908355,0.053968,274.832810,0.385139,...,55.597789,275.814952,100545.634126,0.861225,2.652674,405.7,0.001116,190.803449,214.646028,86.562093
4,NO1,2020-02-15 12:00:00,2020-02-15 16:00:00,4,3.225136,0.475355,0.961101,0.036191,274.208863,0.461787,...,63.683048,275.814952,100545.634126,0.861225,2.652674,405.7,0.001116,183.557128,214.646028,77.712180


In [22]:
data.columns

Index(['region', 'time_ref', 'time', 'lt', 'ws10m_mean', 'ws10m_std',
       'rh2m_mean', 'rh2m_std', 't2m_mean', 't2m_std', 'g10m_mean', 'g10m_std',
       'mslp_mean', 'mslp_std', 'air_temperature_2m',
       'air_pressure_at_sea_level', 'relative_humidity_2m', 'wind_speed_10m',
       'capacity_total', 'precipitation_amount', 'wd10m_mean',
       'wind_direction_10m', 'power_MW'],
      dtype='str')

In [23]:
data.groupby(["region", "time_ref"])["lt"].max().describe()

count    7192.000000
mean       61.670745
std         3.179958
min         4.000000
25%        61.000000
50%        61.000000
75%        64.000000
max        64.000000
Name: lt, dtype: float64

In [24]:
import pandas as pd


def split_by_time_ref(df, train_frac=0.7, val_frac=0.15, test_frac=0.15):
    """
    Splits a forecasting dataset chronologically by time_ref.

    Parameters
    ----------
    df : pd.DataFrame
        Full dataset containing a 'time_ref' column.
    train_frac : float
        Fraction of data used for training.
    val_frac : float
        Fraction used for validation.
    test_frac : float
        Fraction used for testing.

    Returns
    -------
    train_df, val_df, test_df : pd.DataFrame
        Chronological splits.
    """

    assert abs(train_frac + val_frac + test_frac - 1.0) < 1e-6, "Fractions must sum to 1."

    # Sort by issue time
    df = df.sort_values("time_ref").reset_index(drop=True)

    n = len(df)

    train_end = int(train_frac * n)
    val_end = int((train_frac + val_frac) * n)

    train_df = df.iloc[:train_end].copy()
    val_df = df.iloc[train_end:val_end].copy()
    test_df = df.iloc[val_end:].copy()

    print("Split sizes:")
    print("Train:", len(train_df))
    print("Val:  ", len(val_df))
    print("Test: ", len(test_df))

    print("\nTime ranges:")
    print("Train:", train_df["time_ref"].min(), "until", train_df["time_ref"].max())
    print("Val:  ", val_df["time_ref"].min(), "until", val_df["time_ref"].max())
    print("Test: ", test_df["time_ref"].min(), "until", test_df["time_ref"].max())

    return train_df, val_df, test_df

In [25]:
train_df, val_df, test_df = split_by_time_ref(data, train_frac=0.7, val_frac=0.15, test_frac=0.15)

Split sizes:
Train: 315403
Val:   67586
Test:  67587

Time ranges:
Train: 2020-02-15 12:00:00 until 2023-09-26 12:00:00
Val:   2023-09-26 12:00:00 until 2024-06-20 09:00:00
Test:  2024-06-20 09:00:00 until 2025-03-24 09:00:00


In [26]:
train_df.head()

,region,time_ref,time,lt,ws10m_mean,ws10m_std,rh2m_mean,rh2m_std,t2m_mean,t2m_std,...,mslp_std,air_temperature_2m,air_pressure_at_sea_level,relative_humidity_2m,wind_speed_10m,capacity_total,precipitation_amount,wd10m_mean,wind_direction_10m,power_MW
0,NO1,2020-02-15 12:00:00,2020-02-15 12:00:00,0,2.885023,0.564210,0.849861,0.049327,275.620253,0.453982,...,39.606556,275.814952,100545.634126,0.861225,2.652674,405.7,0.001116,218.406386,214.646028,72.052233
1,NO3,2020-02-15 12:00:00,2020-02-15 18:00:00,6,6.831086,0.459264,0.713977,0.038784,274.641646,0.246641,...,79.075508,276.535941,99836.796112,0.842201,6.862173,2268.8,0.729649,134.609069,224.541276,718.864560
2,NO3,2020-02-15 12:00:00,2020-02-15 17:00:00,5,5.953119,0.396569,0.755101,0.035903,274.820247,0.256136,...,70.077424,276.535941,99836.796112,0.842201,6.862173,2268.8,0.729649,136.461239,224.541276,611.849198
3,NO3,2020-02-15 12:00:00,2020-02-15 16:00:00,4,5.172338,0.345095,0.795827,0.035377,275.172954,0.300271,...,64.587791,276.535941,99836.796112,0.842201,6.862173,2268.8,0.729649,142.588408,224.541276,488.289472
4,NO3,2020-02-15 12:00:00,2020-02-15 14:00:00,2,4.778945,0.541424,0.817693,0.043097,276.512762,0.390353,...,55.380598,276.535941,99836.796112,0.842201,6.862173,2268.8,0.729649,189.131557,224.541276,297.167215


In [27]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler


def add_wind_direction_sincos(df, col="wind_direction_10m", prefix="wd"):
    df = df.copy()
    rad = np.deg2rad(df[col].astype(float))
    df[f"{prefix}_sin"] = np.sin(rad)
    df[f"{prefix}_cos"] = np.cos(rad)
    return df


def add_time_sincos(df, time_col="time_ref", prefix="tref"):
    """
    Cyclical encoding of hour-of-day and day-of-year from time_ref.
    """
    df = df.copy()
    t = pd.to_datetime(df[time_col])

    hour = t.dt.hour + t.dt.minute / 60.0
    doy = t.dt.dayofyear.astype(float)

    df[f"{prefix}_hour_sin"] = np.sin(2 * np.pi * hour / 24.0)
    df[f"{prefix}_hour_cos"] = np.cos(2 * np.pi * hour / 24.0)

    df[f"{prefix}_doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
    df[f"{prefix}_doy_cos"] = np.cos(2 * np.pi * doy / 365.25)
    return df


def add_windspeed_polynomials(df, ws_col="ws10m_mean"):
    df = df.copy()
    ws = df[ws_col].astype(float)
    df["ws2"] = ws**2
    df["ws3"] = ws**3
    return df


def add_gust_turbulence(df, ws_col="ws10m_mean", ws_std_col="ws10m_std", gust_col="g10m_mean"):
    df = df.copy()
    ws = df[ws_col].astype(float)
    ws_std = df[ws_std_col].astype(float)
    gust = df[gust_col].astype(float)

    eps = 1e-6
    df["turbulence"] = ws_std / (ws + eps)
    df["gust_factor"] = gust / (ws + eps)
    return df


def add_air_density_proxy(df, p_col="air_pressure_at_sea_level", t_col="air_temperature_2m"):
    df = df.copy()
    p = df[p_col].astype(float)
    t = df[t_col].astype(float)
    df["air_density_proxy"] = p / (t + 1e-6)
    df["ws_x_density"] = df["ws10m_mean"].astype(float) * df["air_density_proxy"]
    return df


def add_lead_time_features(df, lt_col="lt", horizon=61):
    df = df.copy()
    lt = df[lt_col].astype(float)
    df["lt_scaled"] = lt / float(horizon)  # or lt.max() if you prefer
    df["lt_sin"] = np.sin(2 * np.pi * lt / float(horizon))
    df["lt_cos"] = np.cos(2 * np.pi * lt / float(horizon))
    return df


def add_target_normalized_power(df, power_col="power_MW", cap_col="capacity_total", out_col="power_norm"):
    df = df.copy()
    df[out_col] = df[power_col].astype(float) / (df[cap_col].astype(float) + 1e-6)
    return df


class Preprocessor:
    def __init__(self, feature_cols):
        self.feature_cols = feature_cols
        self.scaler = StandardScaler()
        self._fitted = False

    def fit(self, train_df: pd.DataFrame):
        self.scaler.fit(train_df[self.feature_cols])
        self._fitted = True
        return self

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        if not self._fitted:
            raise RuntimeError("Call fit(train_df) before transform(df).")

        df = df.copy()
        df[self.feature_cols] = self.scaler.transform(df[self.feature_cols])
        return df


def apply_feature_engineering(df: pd.DataFrame, horizon=61) -> pd.DataFrame:
    df = df.copy()

    df = add_wind_direction_sincos(df, col="wind_direction_10m", prefix="wd")
    df = add_wind_direction_sincos(df, col="wd10m_mean", prefix="wd_mean")  # optional
    df = add_time_sincos(df, time_col="time_ref", prefix="tref")
    df = add_windspeed_polynomials(df, ws_col="ws10m_mean")
    df = add_gust_turbulence(df)
    df = add_air_density_proxy(df)
    df = add_lead_time_features(df, lt_col="lt", horizon=horizon)

    df = add_target_normalized_power(df, power_col="power_MW", cap_col="capacity_total", out_col="power_norm")

    return df

In [28]:
train_df = apply_feature_engineering(train_df, horizon=61)
val_df = apply_feature_engineering(val_df, horizon=61)
test_df = apply_feature_engineering(test_df, horizon=61)

feature_cols = [
    "ws10m_mean",
    "ws10m_std",
    "rh2m_mean",
    "rh2m_std",
    "t2m_mean",
    "t2m_std",
    "g10m_mean",
    "g10m_std",
    "mslp_mean",
    "mslp_std",
    "air_temperature_2m",
    "air_pressure_at_sea_level",
    "relative_humidity_2m",
    "wind_speed_10m",
    "capacity_total",
    "precipitation_amount",
    "ws2",
    "ws3",
    "turbulence",
    "gust_factor",
    "air_density_proxy",
    "ws_x_density",
    "wd_sin",
    "wd_cos",
    "wd_mean_sin",
    "wd_mean_cos",
    "tref_hour_sin",
    "tref_hour_cos",
    "tref_doy_sin",
    "tref_doy_cos",
    "lt_scaled",
    "lt_sin",
    "lt_cos",
]

prep = Preprocessor(feature_cols).fit(train_df)

train_df = prep.transform(train_df)
val_df = prep.transform(val_df)
test_df = prep.transform(test_df)

In [29]:
train_df.head()

,region,time_ref,time,lt,ws10m_mean,ws10m_std,rh2m_mean,rh2m_std,t2m_mean,t2m_std,...,ws2,ws3,turbulence,gust_factor,air_density_proxy,ws_x_density,lt_scaled,lt_sin,lt_cos,power_norm
0,NO1,2020-02-15 12:00:00,2020-02-15 12:00:00,0,-0.695724,-0.686007,0.244280,0.033694,-0.356976,-0.697347,...,-0.624035,-0.483890,-0.260964,1.546920,0.297474,-0.677386,-1.701864,-0.001565,1.380490,0.177600
1,NO3,2020-02-15 12:00:00,2020-02-15 18:00:00,6,0.892034,-0.894518,-0.755192,-0.356870,-0.491869,-1.210172,...,0.625781,0.327943,-1.436371,-0.662142,-0.035913,0.863413,-1.366616,0.824508,1.120904,0.316848
2,NO3,2020-02-15 12:00:00,2020-02-15 17:00:00,5,0.538771,-1.019085,-0.452714,-0.463590,-0.467250,-1.186688,...,0.259899,0.031063,-1.442015,-0.699256,-0.035913,0.518141,-1.422490,0.700654,1.198450,0.269680
3,NO3,2020-02-15 12:00:00,2020-02-15 16:00:00,4,0.224611,-1.121357,-0.153158,-0.483067,-0.418632,-1.077528,...,-0.023264,-0.168900,-1.441063,-0.631324,-0.035913,0.211088,-1.478365,0.569357,1.263050,0.215219
4,NO3,2020-02-15 12:00:00,2020-02-15 14:00:00,2,0.066324,-0.731278,0.007670,-0.197078,-0.233951,-0.854724,...,-0.150878,-0.249417,-1.014488,0.338646,-0.035913,0.056381,-1.590115,0.290062,1.350816,0.130980


In [30]:
train_df.columns

Index(['region', 'time_ref', 'time', 'lt', 'ws10m_mean', 'ws10m_std',
       'rh2m_mean', 'rh2m_std', 't2m_mean', 't2m_std', 'g10m_mean', 'g10m_std',
       'mslp_mean', 'mslp_std', 'air_temperature_2m',
       'air_pressure_at_sea_level', 'relative_humidity_2m', 'wind_speed_10m',
       'capacity_total', 'precipitation_amount', 'wd10m_mean',
       'wind_direction_10m', 'power_MW', 'wd_sin', 'wd_cos', 'wd_mean_sin',
       'wd_mean_cos', 'tref_hour_sin', 'tref_hour_cos', 'tref_doy_sin',
       'tref_doy_cos', 'ws2', 'ws3', 'turbulence', 'gust_factor',
       'air_density_proxy', 'ws_x_density', 'lt_scaled', 'lt_sin', 'lt_cos',
       'power_norm'],
      dtype='str')

In [31]:
def make_horizon_windows(df, feature_cols, target_col="power_norm", horizon=61):
    df = df.sort_values(["region", "time_ref", "lt"])

    X_list, y_list, meta = [], [], []

    for (region, tref), g in df.groupby(["region", "time_ref"], sort=False):
        g = g.sort_values("lt")

        # take first horizon steps
        g = g.iloc[:horizon]

        if len(g) < horizon:
            continue
        if g[target_col].isna().any():
            continue

        X_list.append(g[feature_cols].to_numpy(dtype=np.float32))
        y_list.append(g[target_col].to_numpy(dtype=np.float32))
        meta.append((region, tref))

    X = np.stack(X_list) if X_list else np.empty((0, horizon, len(feature_cols)), dtype=np.float32)
    y = np.stack(y_list) if y_list else np.empty((0, horizon), dtype=np.float32)

    return X, y, meta

In [32]:
def build_horizon_windows_for_splits(
    train_df,
    val_df,
    test_df,
    feature_cols,
    target_col="power_norm",
    horizon=61,
):
    X_train, y_train, meta_train = make_horizon_windows(
        train_df,
        feature_cols=feature_cols,
        target_col=target_col,
        horizon=horizon,
    )

    X_val, y_val, meta_val = make_horizon_windows(
        val_df,
        feature_cols=feature_cols,
        target_col=target_col,
        horizon=horizon,
    )

    X_test, y_test, meta_test = make_horizon_windows(
        test_df,
        feature_cols=feature_cols,
        target_col=target_col,
        horizon=horizon,
    )

    print("✅ Windows created")
    print(f"  Train: {len(meta_train):,} windows | X {X_train.shape} | y {y_train.shape}")
    print(f"  Val:   {len(meta_val):,} windows | X {X_val.shape}   | y {y_val.shape}")
    print(f"  Test:  {len(meta_test):,} windows | X {X_test.shape}  | y {y_test.shape}")

    return (X_train, y_train, meta_train), (X_val, y_val, meta_val), (X_test, y_test, meta_test)

In [33]:
(train_pack, val_pack, test_pack) = build_horizon_windows_for_splits(
    train_df, val_df, test_df, feature_cols=feature_cols, target_col="power_norm", horizon=61
)

X_train, y_train, meta_train = train_pack

✅ Windows created
  Train: 5,054 windows | X (5054, 61, 33) | y (5054, 61)
  Val:   1,040 windows | X (1040, 61, 33)   | y (1040, 61)
  Test:  1,039 windows | X (1039, 61, 33)  | y (1039, 61)
